# 02 — Face Classification (Supervised Embedding)

Trains two models: CrossEntropy and ArcFace, then evaluates on verification pairs.

In [ ]:
import os
import math
import random
from pathlib import Path

# Use GPU card #2 (index 1). Change to '0' to use the first card.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

DATA_ROOT   = Path("../data")
WEIGHTS_DIR = Path("../weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4000
EMB_DIM     = 512
IMG_SIZE    = 112
BATCH_SIZE  = 64
EPOCHS      = 20
LR          = 1e-3

print(f"Device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

## Model Definitions

In [ ]:
def _build_backbone(backbone='resnet50'):
    base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    layers = list(base.children())[:-1]
    return nn.Sequential(*layers), 2048


class ArcFaceHead(nn.Module):
    def __init__(self, emb_dim, num_classes, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, emb_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        w = F.normalize(self.weight, dim=1)
        cos = F.linear(F.normalize(emb, dim=1), w)
        sin = torch.sqrt(torch.clamp(1 - cos**2, min=1e-7))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.unsqueeze(1).long(), 1.0)
        return self.s * (one_hot * phi + (1 - one_hot) * cos)


class FaceClassifier(nn.Module):
    def __init__(self, num_classes=4000, emb_dim=512):
        super().__init__()
        self.backbone, feat = _build_backbone()
        self.embed = nn.Linear(feat, emb_dim)
        self.bn    = nn.BatchNorm1d(emb_dim)
        self.head  = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        f = self.backbone(x).flatten(1)
        return self.head(self.bn(self.embed(f)))

    def extract_embedding(self, x):
        f = self.backbone(x).flatten(1)
        return F.normalize(self.bn(self.embed(f)), dim=1)


class ArcFaceClassifier(nn.Module):
    def __init__(self, num_classes=4000, emb_dim=512):
        super().__init__()
        self.backbone, feat = _build_backbone()
        self.embed = nn.Linear(feat, emb_dim)
        self.bn    = nn.BatchNorm1d(emb_dim)
        self.head  = ArcFaceHead(emb_dim, num_classes)

    def forward(self, x, labels):
        f = self.backbone(x).flatten(1)
        e = self.bn(self.embed(f))
        return self.head(e, labels)

    def extract_embedding(self, x):
        f = self.backbone(x).flatten(1)
        return F.normalize(self.bn(self.embed(f)), dim=1)


## Data Loading

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(DATA_ROOT / "classification_data/train_data", transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_ROOT / "classification_data/val_data",   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} images, {len(train_ds.classes)} classes")
print(f"Val  : {len(val_ds)}   images, {len(val_ds.classes)}   classes")


## Training Helpers

In [ ]:
def train_epoch_ce(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total


def train_epoch_arc(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs, labels)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(imgs)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, arcface=False):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        emb = model.extract_embedding(imgs)
        # Use cosine classifier for evaluation
        if arcface:
            logits = model(imgs, labels)
        else:
            logits = model(imgs)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += len(imgs)
    return correct / total


## Train — CrossEntropy

In [ ]:
ce_model   = FaceClassifier(NUM_CLASSES, EMB_DIM).to(DEVICE)
ce_optim   = torch.optim.Adam(ce_model.parameters(), lr=LR, weight_decay=1e-4)
ce_sched   = torch.optim.lr_scheduler.CosineAnnealingLR(ce_optim, T_max=EPOCHS)
criterion  = nn.CrossEntropyLoss()

ce_history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch_ce(ce_model, train_loader, ce_optim, criterion)
    val_acc         = eval_epoch(ce_model, val_loader)
    ce_sched.step()
    ce_history["train_loss"].append(tr_loss)
    ce_history["train_acc"].append(tr_acc)
    ce_history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss {tr_loss:.4f} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")

torch.save(ce_model.state_dict(), WEIGHTS_DIR / "face_classification_ce.pth")
print("Saved face_classification_ce.pth")


## Train — ArcFace

In [ ]:
arc_model  = ArcFaceClassifier(NUM_CLASSES, EMB_DIM).to(DEVICE)
arc_optim  = torch.optim.Adam(arc_model.parameters(), lr=LR, weight_decay=1e-4)
arc_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(arc_optim, T_max=EPOCHS)

arc_history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch_arc(arc_model, train_loader, arc_optim, criterion)
    val_acc         = eval_epoch(arc_model, val_loader, arcface=True)
    arc_sched.step()
    arc_history["train_loss"].append(tr_loss)
    arc_history["train_acc"].append(tr_acc)
    arc_history["val_acc"].append(val_acc)
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss {tr_loss:.4f} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")

torch.save(arc_model.state_dict(), WEIGHTS_DIR / "face_classification_arc.pth")
print("Saved face_classification_arc.pth")


## Face Verification Evaluation (ROC / AUC)

In [ ]:
def load_pairs(pairs_file, data_root):
    pairs = []
    with open(pairs_file) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 3:
                pairs.append((data_root / p[0], data_root / p[1], int(p[2])))
    return pairs


@torch.no_grad()
def extract_all_embeddings(model, paths, tf, device):
    model.eval()
    emb_cache = {}
    unique_paths = list(set(paths))
    for p in tqdm(unique_paths, desc="Extracting embeddings"):
        img = tf(Image.open(p).convert("RGB")).unsqueeze(0).to(device)
        emb_cache[p] = model.extract_embedding(img).squeeze(0).cpu().numpy()
    return emb_cache


def compute_pair_scores(emb_cache, pairs):
    cos_scores, euc_scores, labels = [], [], []
    for p1, p2, label in pairs:
        e1, e2 = emb_cache[p1], emb_cache[p2]
        cos_scores.append(float(np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2) + 1e-8)))
        euc_scores.append(-float(np.linalg.norm(e1 - e2)))
        labels.append(label)
    return np.array(cos_scores), np.array(euc_scores), np.array(labels)


pairs = load_pairs(DATA_ROOT / "verification_pairs_val.txt", DATA_ROOT)
all_paths = [p for pair in pairs for p in (pair[0], pair[1])]

results = {}

for name, model in [("CrossEntropy", ce_model), ("ArcFace", arc_model)]:
    emb_cache = extract_all_embeddings(model, all_paths, val_tf, DEVICE)
    cos_s, euc_s, lbls = compute_pair_scores(emb_cache, pairs)
    results[name] = {
        "cosine_auc":    roc_auc_score(lbls, cos_s),
        "euclidean_auc": roc_auc_score(lbls, euc_s),
        "cos_roc":  roc_curve(lbls, cos_s),
        "euc_roc":  roc_curve(lbls, euc_s),
    }
    print(f"{name} — cosine AUC: {results[name]['cosine_auc']:.4f} | euclidean AUC: {results[name]['euclidean_auc']:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {"CrossEntropy": ("steelblue", "royalblue"), "ArcFace": ("tomato", "firebrick")}
for name, res in results.items():
    fpr, tpr, _ = res["cos_roc"]
    ax.plot(fpr, tpr, color=colors[name][0], label=f"{name} cosine  AUC={res['cosine_auc']:.4f}")
    fpr, tpr, _ = res["euc_roc"]
    ax.plot(fpr, tpr, color=colors[name][1], linestyle="--", label=f"{name} euclidean AUC={res['euclidean_auc']:.4f}")

ax.plot([0,1],[0,1], 'k--', alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC Curves — Face Verification")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## Save Best Model

In [ ]:
# Save the ArcFace model as the main face_classification.pth (used by inference)
best_ce_auc  = max(results["CrossEntropy"]["cosine_auc"], results["CrossEntropy"]["euclidean_auc"])
best_arc_auc = max(results["ArcFace"]["cosine_auc"],      results["ArcFace"]["euclidean_auc"])

if best_arc_auc >= best_ce_auc:
    best_model = arc_model
    print(f"Best: ArcFace (AUC {best_arc_auc:.4f})")
else:
    best_model = ce_model
    print(f"Best: CrossEntropy (AUC {best_ce_auc:.4f})")

torch.save(best_model.state_dict(), WEIGHTS_DIR / "face_classification.pth")
print("Saved face_classification.pth")
